<a href="https://colab.research.google.com/github/Frank1to/Proyecto-Ecommerce-/blob/main/Proyecto_MultiAgente_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema Multi-Agente: E-Commerce Product Intelligence
Este notebook implementa una arquitectura avanzada de tres agentes para el análisis de interacciones en comercio electrónico.

### Requisitos técnicos:
- Agente 1: Normalización + **Transformers** (Embeddings) + Vector DB (FAISS).
- Agente 2: Selección inteligente de modelos (Meta-Learning).
- Agente 3: Comunicador con NLP profesional.

## 1. Instalación de Dependencias

In [1]:
!pip install sentence-transformers faiss-cpu transformers joblib python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 32.5 MB/s eta 0:00:00


## 2. Carga del Dataset
Ejecuta esta celda y sube tu archivo `interactions.csv`.

In [2]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'Archivo {filename} cargado con éxito!')

Saving requirements.txt to requirements.txt
Archivo requirements.txt cargado con éxito!


## 3. Implementación de los Agentes

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
import faiss

# AGENTE 1: Especialista en Datos y Vectores
class NormalizerAgent:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.encoder = SentenceTransformer(model_name)
        self.scaler = StandardScaler()
        self.label_encoders = {}

    def process(self, df, target_col='interaction_type'):
        df = df.drop_duplicates()
        df['dwell_time_ms'] = df['dwell_time_ms'].fillna(df['dwell_time_ms'].median())

        # Características temporales
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour'] = df['timestamp'].dt.hour
        df['day_of_week'] = df['timestamp'].dt.dayofweek

        # Embeddings con Transformers
        text_data = df.apply(lambda x: f"User {x['user_id']} {x['interaction_type']} product {x['product_id']}", axis=1).tolist()
        embeddings = self.encoder.encode(text_data, show_progress_bar=True)

        # Scaling
        features = ['dwell_time_ms', 'hour', 'day_of_week']
        df[features] = self.scaler.fit_transform(df[features])

        # Encoding
        le = LabelEncoder()
        df[target_col] = le.fit_transform(df[target_col])

        return df, np.array(embeddings), le

# AGENTE 2: Entrenador y Selector Inteligente
class TrainerAgent:
    def __init__(self):
        self.models = {
            "LogisticRegression": LogisticRegression(max_iter=1000),
            "RandomForest": RandomForestClassifier(n_estimators=100),
            "GradientBoosting": HistGradientBoostingClassifier()
        }

    def select_best(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        results = {}
        for name, model in self.models.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            results[name] = {"f1": f1_score(y_test, y_pred, average='weighted'), "model": model, "acc": accuracy_score(y_test, y_pred)}

        best_name = max(results, key=lambda k: results[k]['f1'])
        return best_name, results[best_name]

# AGENTE 3: Comunicador Profesional
class CommunicatorAgent:
    def generate(self, model_name, metrics):
        report = f"""
# Reporte Final: Inteligencia de E-Commerce

## Resumen Técnico
El sistema de agentes ha procesado el dataset utilizando **Transformers** para la representación semántica.

**Modelo Seleccionado:** {model_name}
**Métricas:**
- Accuracy: {metrics['acc']:.2%}
- F1-Score: {metrics['f1']:.4f}

## Conclusión
La arquitectura de vectores (FAISS) y embeddings permite una personalización avanzada de las interacciones.
"""
        return report

## 4. Ejecución del Pipeline Multi-Agente

In [5]:
# 1. Agente 1
ag1 = NormalizerAgent()
df_clean, embeddings, target_le = ag1.process(df)

# 2. Agente 2
X = df_clean[['dwell_time_ms', 'hour', 'day_of_week']]
y = df_clean['interaction_type']
ag2 = TrainerAgent()
best_name, best_metrics = ag2.select_best(X, y)

# 3. Agente 3
ag3 = CommunicatorAgent()
final_report = ag3.generate(best_name, best_metrics)

print("PIPELINE COMPLETADO CON ÉXITO")
from IPython.display import Markdown
Markdown(final_report)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

KeyError: 'dwell_time_ms'